##CSV and EXcel files - Stuctured Data 

In [5]:
import pandas 
import os

In [3]:
os.makedirs('data/structured_files', exist_ok=True)

In [7]:

import pandas as pd

# Create the data dictionary
data = {
    'Product': ['Laptop', 'Mouse', 'Keyboard', 'Monitor', 'Webcam'],
    'Category': ['Electronics', 'Accessories', 'Electronics', 'Electronics', 'Electronics'],
    'Price': [999.99, 29.99, 79.99, 299.99, 89.99],
    'Stock': [50, 200, 150, 75, 100],
    'Description': [
        'High-performance laptop with 16GB RAM and 512GB SSD',
        'Wireless optical mouse with ergonomic design',
        'Mechanical keyboard with RGB backlighting',
        '27-inch 4K monitor with HDR support',
        '1080p webcam with noise cancellation'
    ]
}

# Convert to pandas DataFrame
df = pd.DataFrame(data)
df.to_csv('data/structured_files/products.csv', index=False)

In [8]:
# Save as Excel with multiple sheets
with pd.ExcelWriter('data/structured_files/inventory.xlsx') as writer:
    df.to_excel(writer, sheet_name='Products', index=False)

# Add another sheet
summary_data = {
    'Category': ['Electronics', 'Accessories'],
    'Total_Items': [3, 2],
    'Total_Value': [1389.97, 109.98]
}

pd.DataFrame(summary_data).to_excel(writer, sheet_name='Summary', index=False)

##CSV Processing 

In [1]:
from langchain_community.document_loaders import CSVLoader, UnstructuredCSVLoader


C:\Users\Manish\AppData\Local\Temp\ipykernel_20768\2855580462.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import CSVLoader, UnstructuredCSVLoader
c:\Users\Manish\OneDrive\Desktop\MANISH_2026\AI_WORKSETUP\langchain_init_proj\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Method 1: CSVLoader - Each row becomes a document
# This loads a CSV file where each row becomes a separate document
print("CSVLoader - Row-based Documents")

# Create a CSVLoader instance with file path and encoding settings
csv_loader = CSVLoader(
    file_path='data/structured_files/products.csv',  # Path to CSV file
    encoding='utf-8',  # File encoding
    csv_args={  # CSV parsing arguments
        'delimiter': ',',  # Column separator
        'quotechar': '"',  # Character for quoting fields
    }
)

# Load the CSV - each row becomes a Document object
# Each document's content is a string representation of the row
csv_docs = csv_loader.load()

# Display loading results
print(f"Loaded {len(csv_docs)} documents (one per row)")

# Show first document as example
print("\nFirst document:")
print(f"Content: {csv_docs[0].page_content}")  # The row data as string
print(f"Metadata: {csv_docs[0].metadata}")    # Source information

CSVLoader - Row-based Documents
Loaded 5 documents (one per row)

First document:
Content: Product: Laptop
Category: Electronics
Price: 999.99
Stock: 50
Description: High-performance laptop with 16GB RAM and 512GB SSD
Metadata: {'source': 'data/structured_files/products.csv', 'row': 0}


In [7]:
from typing import List
from langchain_core.documents import Document
import pandas as pd

# Method 2: Custom CSV processing for better control
print("\n2 Custom CSV Processing")

def process_csv_intelligently(filepath: str) -> List[Document]:
    """Process CSV with intelligent document creation"""
    
    # Read CSV file into pandas DataFrame
    df = pd.read_csv(filepath)
    documents = []

    # Strategy 1: One document per row with structured content
    for idx, row in df.iterrows():
        # Create structured content with clear formatting
        content = f"""Product Information:
Name: {row['Product']}
Category: {row['Category']}
Price: ${row['Price']}
Stock: {row['Stock']} units
Description: {row['Description']}"""
        
        # Create Document object with content and metadata (rich metadata)
        doc = Document(
            page_content=content,
            metadata={
                'source': filepath,
                'row_index': idx,
                'product': row['Product'],
                'category': row['Category'],
                'price': row['Price'],
                'data_type': 'product_info'
            }
        )
        documents.append(doc)
    
    return documents

# Usage
csv_docs_custom = process_csv_intelligently('data/structured_files/products.csv')

print(f"Loaded {len(csv_docs_custom)} documents")
print("\nFirst document:")
print(f"Content: {csv_docs_custom[0].page_content}")
print(f"Metadata: {csv_docs_custom[0].metadata}")


2 Custom CSV Processing
Loaded 5 documents

First document:
Content: Product Information:
Name: Laptop
Category: Electronics
Price: $999.99
Stock: 50 units
Description: High-performance laptop with 16GB RAM and 512GB SSD
Metadata: {'source': 'data/structured_files/products.csv', 'row_index': 0, 'product': 'Laptop', 'category': 'Electronics', 'price': 999.99, 'data_type': 'product_info'}


In [5]:
process_csv_intelligently('data/structured_files/products.csv')


[Document(metadata={'source': 'data/structured_files/products.csv', 'row_index': 0, 'product': 'Laptop', 'category': 'Electronics'}, page_content='Product Information:\nName: Laptop\nCategory: Electronics\nPrice: $999.99\nStock: 50 units\nDescription: High-performance laptop with 16GB RAM and 512GB SSD'),
 Document(metadata={'source': 'data/structured_files/products.csv', 'row_index': 1, 'product': 'Mouse', 'category': 'Accessories'}, page_content='Product Information:\nName: Mouse\nCategory: Accessories\nPrice: $29.99\nStock: 200 units\nDescription: Wireless optical mouse with ergonomic design'),
 Document(metadata={'source': 'data/structured_files/products.csv', 'row_index': 2, 'product': 'Keyboard', 'category': 'Electronics'}, page_content='Product Information:\nName: Keyboard\nCategory: Electronics\nPrice: $79.99\nStock: 150 units\nDescription: Mechanical keyboard with RGB backlighting'),
 Document(metadata={'source': 'data/structured_files/products.csv', 'row_index': 3, 'product':

In [8]:
# CSV Processing Strategies:
# =========================

# Strategy 1: Row-based (CSVLoader)
# - ✅ Simple one-row-one-document
# - ✅ Good for record lookups
# - ❌ Loses table context

print("\nStrategy 1: Row-based CSVLoader")
print("-" * 50)
from langchain_community.document_loaders import CSVLoader

csv_loader = CSVLoader(
    file_path='data/structured_files/products.csv',
    encoding='utf-8',
    csv_args={
        'delimiter': ',',
        'quotechar': '"',
    }
)

csv_docs = csv_loader.load()
print(f"Loaded {len(csv_docs)} documents (one per row)")
print(f"Example: {csv_docs[0].page_content[:100]}...")

# Strategy 2: Intelligent Processing
# - ✅ Preserves relationships
# - ✅ Creates summaries
# - ✅ Rich metadata
# - ✅ Better for Q&A

print("\nStrategy 2: Intelligent Processing")
print("-" * 50)

def process_csv_intelligently(filepath: str):
    """Process CSV with intelligent document creation"""
    import pandas as pd
    from langchain_core.documents import Document
    from typing import List
    
    df = pd.read_csv(filepath)
    documents = []
    
    # Create summary statistics for context
    total_products = len(df)
    categories = df['Category'].unique().tolist()
    avg_price = df['Price'].mean()
    total_stock = df['Stock'].sum()
    
    # Add summary document
    summary_content = f"""CSV Summary:
- Total Products: {total_products}
- Categories: {', '.join(categories)}
- Average Price: ${avg_price:.2f}
- Total Stock: {total_stock} units
- Price Range: ${df['Price'].min():.2f} - ${df['Price'].max():.2f}"""
    
    summary_doc = Document(
        page_content=summary_content,
        metadata={
            'source': filepath,
            'type': 'summary',
            'total_products': total_products,
            'categories': categories
        }
    )
    documents.append(summary_doc)
    
    # Process each row with rich context
    for idx, row in df.iterrows():
        content = f"""Product Information:
Name: {row['Product']}
Category: {row['Category']}
Price: ${row['Price']}
Stock: {row['Stock']} units
Description: {row['Description']}

Category Context: {row['Category']} product
Stock Status: {'Low Stock' if row['Stock'] < 100 else 'In Stock'}
Price Level: {'Premium' if row['Price'] > 500 else 'Standard'}"""
        
        doc = Document(
            page_content=content,
            metadata={
                'source': filepath,
                'type': 'product',
                'row_index': idx,
                'product': row['Product'],
                'category': row['Category'],
                'price': row['Price'],
                'stock': row['Stock']
            }
        )
        documents.append(doc)
    
    return documents

# Use intelligent processing
intelligent_docs = process_csv_intelligently('data/structured_files/products.csv')
print(f"Loaded {len(intelligent_docs)} documents")
print(f"Document 1 (Summary): {intelligent_docs[0].page_content[:150]}...")
print(f"Document 2 (Product): {intelligent_docs[1].page_content[:150]}...")
print(f"Metadata: {intelligent_docs[1].metadata}")

# Comparison
print("\nComparison Summary:")
print("=" * 50)
print("CSVLoader (Row-based):")
print("  - Loaded: 5 documents")
print("  - Each document: One row of data")
print("  - Use case: Simple record retrieval")

print("\nIntelligent Processing:")
print(f"  - Loaded: {len(intelligent_docs)} documents (1 summary + 5 products)")
print("  - Each document: Enriched with context and metadata")
print("  - Use case: Q&A, analysis, and enhanced retrieval")


Strategy 1: Row-based CSVLoader
--------------------------------------------------
Loaded 5 documents (one per row)
Example: Product: Laptop
Category: Electronics
Price: 999.99
Stock: 50
Description: High-performance laptop w...

Strategy 2: Intelligent Processing
--------------------------------------------------
Loaded 6 documents
Document 1 (Summary): CSV Summary:
- Total Products: 5
- Categories: Electronics, Accessories
- Average Price: $299.99
- Total Stock: 575 units
- Price Range: $29.99 - $999...
Document 2 (Product): Product Information:
Name: Laptop
Category: Electronics
Price: $999.99
Stock: 50 units
Description: High-performance laptop with 16GB RAM and 512GB SS...
Metadata: {'source': 'data/structured_files/products.csv', 'type': 'product', 'row_index': 0, 'product': 'Laptop', 'category': 'Electronics', 'price': 999.99, 'stock': 50}

Comparison Summary:
CSVLoader (Row-based):
  - Loaded: 5 documents
  - Each document: One row of data
  - Use case: Simple record retrieval

##EXcel Processing 

In [9]:
# Method 1: Using pandas for full control
print("Pandas-based Excel Processing")

def process_excel_with_pandas(filepath: str) -> List[Document]:
    """Process Excel with sheet awareness"""
    documents = []

    # Read all sheets
    excel_file = pd.ExcelFile(filepath)

    for sheet_name in excel_file.sheet_names:
        df = pd.read_excel(filepath, sheet_name=sheet_name)

        # Create document for each sheet
        sheet_content = f"Sheet: {sheet_name}\n"
        sheet_content += f"Columns: {', '.join(df.columns)}\n"
        sheet_content += f"Rows: {len(df)}\n\n"
        sheet_content += df.to_string(index=False)

        doc = Document(
            page_content=sheet_content,
            metadata={
                'source': filepath,
                'sheet_name': sheet_name,
                'num_rows': len(df),
                'num_columns': len(df.columns),
                'data_type': 'excel_sheet'
            }
        )

        documents.append(doc)

    return documents

Pandas-based Excel Processing


In [10]:
# Method 2: Using UnstructuredExcelLoader
print("\n2 UnstructuredExcelLoader")

try:
    # Create UnstructuredExcelLoader instance
    # mode="elements" extracts each element separately (tables, text, etc.)
    excel_loader = UnstructuredExcelLoader(
        'data/structured_files/inventory.xlsx',
        mode="elements"  # 'elements' mode preserves document structure
    )
    
    # Load the Excel file
    unstructured_docs = excel_loader.load()
    
    print("✅ Handles created")
    print("✅ Preserves data structure")
    print("✅ Requires unstructured library")
    
    # Display loaded documents
    print(f"\nLoaded {len(unstructured_docs)} documents")
    
    # Show first document as example
    if unstructured_docs:
        print("\nFirst document:")
        print(f"Content: {unstructured_docs[0].page_content[:200]}...")
        print(f"Metadata: {unstructured_docs[0].metadata}")
    
except Exception as e:
    print("❌ Error: Unstructured library with Excel support required")
    print("   Install with: pip install unstructured")
    print(f"   Error details: {e}")


2 UnstructuredExcelLoader
❌ Error: Unstructured library with Excel support required
   Install with: pip install unstructured
   Error details: name 'UnstructuredExcelLoader' is not defined
